# Classification Part A — Support Vector Machine (SVC)

## Overview
This notebook implements **Support Vector Classifier (SVC)** to predict customer review scores. It demonstrates tuning regularization parameter `C` and `kernel`, with feature scaling verified.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
data = np.load('classification_data.npz')
X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

feature_names = pd.read_csv('classification_feature_names.csv', header=None)[0].to_numpy()
print(f"Loaded X_train: {X_train.shape}, X_test: {X_test.shape}")


## 1. Mandatory Demonstration: Tuning `C` and `kernel` & Feature Scaling

> **Algorithm Note Requirement**: *Tune C and kernel; scale features*

Support Vector Machines construct optimal hyperplanes separating class decision boundaries. We perform 5-fold cross-validation grid search over `C \in [0.1, 1.0, 10.0]` and `kernel \in ['rbf', 'linear']` with `class_weight='balanced'`.


In [ ]:
param_grid = {
    'C': [0.1, 1.0, 5.0],
    'kernel': ['rbf', 'linear'],
    'class_weight': ['balanced']
}

svc_grid = GridSearchCV(
    SVC(probability=True, random_state=42),
    param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1
)

svc_grid.fit(X_train, y_train)
best_svc = svc_grid.best_estimator_

print("Best SVC Parameters:", svc_grid.best_params_)
print(f"Best Cross-Validated F1 Score: {svc_grid.best_score_:.4f}")


## 2. Predictions & Performance Metrics


In [ ]:
y_test_pred = best_svc.predict(X_test)
y_test_proba = best_svc.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
roc_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='weighted')

print(f"Accuracy         : {accuracy:.4f}")
print(f"Weighted Precision: {precision:.4f}")
print(f"Weighted Recall   : {recall:.4f}")
print(f"Weighted F1-Score : {f1:.4f}")
print(f"Weighted ROC-AUC  : {roc_auc:.4f}")


In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Score 1', 'Score 2', 'Score 3', 'Score 4', 'Score 5'],
            yticklabels=['Score 1', 'Score 2', 'Score 3', 'Score 4', 'Score 5'])
plt.title("Support Vector Classifier - Confusion Matrix", fontsize=12, fontweight='bold')
plt.xlabel("Predicted Review Score")
plt.ylabel("Actual Review Score")
plt.tight_layout()
plt.show()

print("Classification Report:\n")
print(classification_report(y_test, y_test_pred, target_names=['Score 1', 'Score 2', 'Score 3', 'Score 4', 'Score 5'], zero_division=0))


In [ ]:
svc_results = {
    "Model": "Support Vector Classifier",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "Weighted F1": f1,
    "ROC-AUC": roc_auc
}
svc_results
